# Exploración y Verificación de la Base de Datos

Este notebook sirve para inspeccionar el flujo completo de datos desde Google Sheets hasta la base de datos SQLite y verificar que la información esté estructurada correctamente.

In [29]:
# 1. Configuración e importación de dependencias
import sys
import os
import pandas as pd
import sqlite3

# Asegurar que podamos importar módulos de src/
sys.path.append(os.path.abspath('.'))
from src.fetcher import fetch_sheet_csv
from src.parser import parse_ejercicios, parse_ciclo
from src.database import init_db, load_ejercicios, load_training_data
from config import DB_PATH

### 2. Descarga y parsing de datos reales

In [30]:
# Descargar CSVs de Google Sheets
print("Descargando datos...")
csv_ejercicios = fetch_sheet_csv("ejercicios")
csv_ciclo = fetch_sheet_csv("ciclo_16")

# Parsear datos
df_ej = parse_ejercicios(csv_ejercicios)
df_ciclo = parse_ciclo(csv_ciclo)

print(f"Catálogo de ejercicios: {len(df_ej)} registros parseados.")
print(f"Ciclo de entrenamiento: {len(df_ciclo)} series parseadas.")

Descargando datos...
Catálogo de ejercicios: 26 registros parseados.
Ciclo de entrenamiento: 1814 series parseadas.


### 3. Inspeccionar DataFrames

In [31]:
print("=== MUESTRA DEL CATALOGO ===")
display(df_ej.head(10))

print("=== MUESTRA DE LAS SERIES DE ENTRENAMIENTO ===")
display(df_ciclo.head(10))

=== MUESTRA DEL CATALOGO ===


,grupo_muscular,ejercicio
0,Pectoral,Press Convergente
1,Pectoral,Press Mancuernas
2,Pectoral,contracción Pectoral
3,Triceps,Katana
4,Triceps,Extención Tricps
5,Hombro,Elevaciones Lateral
6,Hombro,Press Militar
7,Hombro,Pajaros Polea
8,Biceps,Curl Bayesian
9,Biceps,Curl Scot


=== MUESTRA DE LAS SERIES DE ENTRENAMIENTO ===


,semana,dia,fecha,set_orden,ejercicio,reps,kg,rir
0,1,LUNES,4/5/26,1,Press Convergente,6.0,85.0,1.0
1,2,LUNES,11/5/26,1,Press Convergente,6.0,90.0,1.0
2,3,LUNES,18/5/26,1,Press Convergente,7.0,90.0,0.5
3,4,LUNES,25/5/26,1,Press Convergente,5.0,90.0,0.5
4,5,LUNES,1/6/26,1,Press Convergente,6.0,90.0,0.8
5,6,LUNES,8/6/26,1,Press Mancuernas,7.0,32.0,0.8
6,7,LUNES,15/6/26,1,Press Convergente,6.0,90.0,0.8
7,8,LUNES,22/6/26,1,Press Convergente,6.0,90.0,NaN
8,9,LUNES,29/6/26,1,Curl Bayesian,7.0,14.0,1.2
9,10,LUNES,6/7/26,1,Press Mancuernas,6.0,32.0,1.3


### 4. Carga en SQLite
Inicializamos la base de datos de pruebas (o la del sistema) y cargamos los datos.

In [32]:
test_db = "data/test_exploration.db"
os.makedirs("data", exist_ok=True)

print(f"Inicializando base de datos en: {test_db}")
init_db(test_db)
load_ejercicios(test_db, df_ej)
load_training_data(test_db, df_ciclo)
print("¡Datos cargados con éxito!")

Inicializando base de datos en: data/test_exploration.db
¡Datos cargados con éxito!


### 5. Verificación de Calidad de Datos (Calidad del Parser)

Verificamos que los nombres de los ejercicios no contengan valores numéricos residuales.

In [33]:
conn = sqlite3.connect(test_db)

# 1. Listar ejercicios únicos registrados en los entrenamientos
ejercicios_unicos = pd.read_sql_query(
    "SELECT DISTINCT ejercicio FROM training_sets ORDER BY ejercicio", 
    conn
)

print(f"Ejercicios únicos detectados en entrenamiento: {len(ejercicios_unicos)}")
print("Lista de ejercicios registrados:")
for idx, row in ejercicios_unicos.iterrows():
    print(f" - {row['ejercicio']}")

conn.close()

Ejercicios únicos detectados en entrenamiento: 24
Lista de ejercicios registrados:
 - Aducción
 - Curl Bayesian
 - Curl Scot
 - Elevaciones Lateral
 - Elevación Gemelos
 - Exten Cuadriceps
 - Extención Tricps
 - Hack
 - Isquio Acostado
 - Isquio Sentado
 - Jalon Vertical
 - Katana
 - PM Rigidas
 - Pajaros Polea
 - Prensa
 - Press Convergente
 - Press Mancuernas
 - Press Militar
 - Pull Over
 - Remo Neutro
 - Remo Prono
 - Rueda Abdominal
 - Sentadilla Bulgara
 - contracción Pectoral


### 6. Consultar Progresión de un Ejercicio de Ejemplo

In [34]:
conn = sqlite3.connect(test_db)

ejercicio_ejemplo = "Press Convergente"
df_prog = pd.read_sql_query("""
    SELECT *
    FROM training_sets
    WHERE ejercicio = ? AND kg IS NOT NULL 
    ORDER BY semana
""", conn, params=[ejercicio_ejemplo])

print(f"Progresión para: {ejercicio_ejemplo}")
display(df_prog)

conn.close()

Progresión para: Press Convergente


,id,semana,dia,fecha,set_orden,ejercicio,reps,kg,rir
0,1,1,LUNES,4/5/26,1,Press Convergente,6.0,85.0,1.0
1,22,1,LUNES,4/5/26,2,Press Convergente,6.0,85.0,0.0
2,1078,1,VIERNES,8/5/26,64,Press Convergente,6.0,85.0,NaN
3,1099,1,VIERNES,8/5/26,65,Press Convergente,6.0,85.0,NaN
4,2,2,LUNES,11/5/26,1,Press Convergente,6.0,90.0,1.0
5,23,2,LUNES,11/5/26,2,Press Convergente,6.0,90.0,0.4
6,1079,2,VIERNES,15/5/26,64,Press Convergente,7.0,90.0,0.4
7,1100,2,VIERNES,15/5/26,65,Press Convergente,6.0,90.0,NaN
8,3,3,LUNES,18/5/26,1,Press Convergente,7.0,90.0,0.5
9,24,3,LUNES,18/5/26,2,Press Convergente,6.0,90.0,0.5
